In [ ]:
!ls /kaggle/input/datasets/
!echo "--- SC V2 sample ---"
!ls /kaggle/input/datasets/sylkaladin/speech-commands-v2/ | head -5
!echo "--- MUSAN root ---"
!ls /kaggle/input/datasets/nhattruongdev/musan-noise/musan/
!echo "--- RIRs ---"
!ls /kaggle/input/datasets/malaymishra6969/rirssmall/rirs_small/ | head -5


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
REPO_SHA = "06b9e82"
!rm -rf /kaggle/working/srib
!git clone https://github.com/MalayM09/SRIB.git /kaggle/working/srib
!cd /kaggle/working/srib && git checkout $REPO_SHA
%cd /kaggle/working/srib/code


In [ ]:
!pip install -q einops==0.8.0 soundfile==0.12.1 librosa==0.10.2.post1 \
               pyyaml==6.0.2 tqdm==4.66.4 wandb==0.17.7 pytest==8.3.2


In [ ]:
!rm -rf /kaggle/working/srib/code/data
!mkdir -p /kaggle/working/srib/code/data

!ln -s /kaggle/input/datasets/sylkaladin/speech-commands-v2         /kaggle/working/srib/code/data/speech_commands_v2
!ln -s /kaggle/input/datasets/nhattruongdev/musan-noise/musan       /kaggle/working/srib/code/data/musan_small
!ln -s /kaggle/input/datasets/malaymishra6969/rirssmall/rirs_small  /kaggle/working/srib/code/data/rirs_small

print("--- SC V2 ---")
!ls /kaggle/working/srib/code/data/speech_commands_v2/ | head -5
print("--- MUSAN (must show noise/ and music/) ---")
!ls /kaggle/working/srib/code/data/musan_small/
print("--- MUSAN noise files ---")
!ls /kaggle/working/srib/code/data/musan_small/noise/ | head -3
print("--- RIRs ---")
!ls /kaggle/working/srib/code/data/rirs_small/ | head -5


In [ ]:
import torch, torchaudio
print("torch     :", torch.__version__)
print("torchaudio:", torchaudio.__version__)
print("cuda      :", torch.cuda.is_available(), "| n_gpu:", torch.cuda.device_count())
print("device 0  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")


In [ ]:
!cd /kaggle/working/srib/code && python -m pytest -q tests/test_shapes.py


In [ ]:
!cd /kaggle/working/srib/code && python -m src.train --config configs/pilot_p0_baseline.yaml


In [ ]:
import json
m = json.load(open("/kaggle/working/runs/p0/metrics.json"))
print("P0 best_val_acc:", round(m["best_val_acc"], 4))
print("P0 final epoch :", m["history"][-1])
!tar czf /kaggle/working/p0_bundle.tar.gz -C /kaggle/working/runs p0
!ls -lh /kaggle/working/p0_bundle.tar.gz


In [ ]:
!cd /kaggle/working/srib/code && python -m src.train --config configs/pilot_p1_pcen_ablation.yaml


In [ ]:
import json
m = json.load(open("/kaggle/working/runs/p1_pcen_on/metrics.json"))
print("PCEN-ON best_val_acc:", round(m["best_val_acc"], 4))
print("  final:", m["history"][-1])


In [ ]:
LOGMEL_CFG = """run:
  name: p1_logmel
  seed: 1337
  device: cuda
  out_dir: /kaggle/working/runs/p1_logmel

data:
  root: data/speech_commands_v2
  batch_size: 256
  num_workers: 4
  sample_rate: 16000

model:
  trunk: bc_resnet8
  use_pcen: false
  n_mels: 40
  num_classes: 12

optim:
  lr: 4.0e-3
  weight_decay: 1.0e-4
  epochs: 25
  scheduler: cosine
  warmup_epochs: 2

augment:
  specaugment: true
  rir_prob: 0.5
  musan_prob: 0.7
  snr_range: [-5, 20]
"""
with open("/kaggle/working/srib/code/configs/pilot_p1_logmel.yaml", "w") as f:
    f.write(LOGMEL_CFG)
!cat /kaggle/working/srib/code/configs/pilot_p1_logmel.yaml


In [ ]:
!cd /kaggle/working/srib/code && python -m src.train --config configs/pilot_p1_logmel.yaml


In [ ]:
import json
m = json.load(open("/kaggle/working/runs/p1_logmel/metrics.json"))
print("log-Mel best_val_acc:", round(m["best_val_acc"], 4))
print("  final:", m["history"][-1])


In [ ]:
EVAL_SNR = '''"""SNR-sweep eval: mix MUSAN noise at fixed SNRs over the val set."""
from __future__ import annotations
import argparse, csv, random
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from torch.utils.data import DataLoader

from src.data import SpeechCommandsV2
from src.data.augment import _load_wavs, _rms, SAMPLE_RATE
from src.train import KWSModel


def build_noise_paths(musan_dir: Path):
    return _load_wavs(musan_dir / "noise") + _load_wavs(musan_dir / "music")


def mix_at_snr(wav: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    if len(noise) < len(wav):
        reps = (len(wav) // len(noise)) + 1
        noise = np.tile(noise, reps)
    noise = noise[:len(wav)].astype(np.float32)
    sig_rms = _rms(wav)
    noise_rms = _rms(noise)
    if noise_rms < 1e-6 or sig_rms < 1e-6:
        return wav
    target_noise_rms = sig_rms / (10 ** (snr_db / 20.0))
    noise = noise * (target_noise_rms / noise_rms)
    out = (wav + noise).astype(np.float32)
    peak = float(np.max(np.abs(out)) + 1e-9)
    if peak > 0.99:
        out = out * (0.99 / peak)
    return out


@torch.no_grad()
def eval_at_snr(model, loader, noise_paths, snr_db, device, seed):
    rng = random.Random(seed)
    total, correct = 0, 0
    for wav_batch, label_batch in loader:
        if snr_db < 900 and noise_paths:
            mixed = []
            for w in wav_batch:
                w_np = w.numpy()
                tries = 0
                noise = None
                while tries < 5:
                    path = rng.choice(noise_paths)
                    try:
                        n, sr = sf.read(str(path))
                        if n.ndim > 1: n = n[:, 0]
                        if sr == SAMPLE_RATE:
                            noise = n.astype(np.float32)
                            break
                    except Exception:
                        pass
                    tries += 1
                if noise is None:
                    mixed.append(w); continue
                if len(noise) > len(w_np):
                    start = rng.randint(0, len(noise) - len(w_np))
                    noise = noise[start:start + len(w_np)]
                out = mix_at_snr(w_np, noise, snr_db)
                mixed.append(torch.from_numpy(out))
            wav_batch = torch.stack(mixed)
        wav_batch = wav_batch.to(device, non_blocking=True)
        label_batch = label_batch.to(device, non_blocking=True)
        logits = model(wav_batch)
        total += label_batch.size(0)
        correct += (logits.argmax(-1) == label_batch).sum().item()
    return correct / total


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--musan_dir", default="data/musan_small")
    ap.add_argument("--out", required=True)
    ap.add_argument("--snrs", default="999,20,10,5,0,-5")
    ap.add_argument("--seed", type=int, default=1337)
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(args.ckpt, map_location=device, weights_only=False)
    cfg = ckpt["cfg"]; mcfg = cfg["model"]

    model = KWSModel(
        n_mels=mcfg["n_mels"],
        num_classes=mcfg["num_classes"],
        use_pcen=mcfg.get("use_pcen", True),
        specaug=False,
        sample_rate=cfg["data"]["sample_rate"],
    ).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()

    val_ds = SpeechCommandsV2(cfg["data"]["root"], split="val", seed=args.seed)
    loader = DataLoader(val_ds, batch_size=cfg["data"]["batch_size"], shuffle=False, num_workers=2)
    noise_paths = build_noise_paths(Path(args.musan_dir))
    print(f"loaded ckpt={args.ckpt} | val={len(val_ds)} | noise_files={len(noise_paths)}")

    snrs = [float(s) for s in args.snrs.split(",")]
    results = []
    for s in snrs:
        acc = eval_at_snr(model, loader, noise_paths, s, device, args.seed)
        label = "clean" if s >= 900 else f"{int(s)}dB"
        print(f"  SNR={label:>6s}  val_acc={acc:.4f}")
        results.append({"snr_db": s, "val_acc": acc})

    out = Path(args.out); out.parent.mkdir(parents=True, exist_ok=True)
    with out.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["snr_db", "val_acc"])
        w.writeheader(); w.writerows(results)
    print(f"wrote {out}")


if __name__ == "__main__":
    main()
'''
with open("/kaggle/working/srib/code/src/eval_snr.py", "w") as f:
    f.write(EVAL_SNR)
print("wrote /kaggle/working/srib/code/src/eval_snr.py")


In [ ]:
!cd /kaggle/working/srib/code && python -m src.eval_snr \
    --ckpt /kaggle/working/runs/p1_pcen_on/best.pt \
    --musan_dir data/musan_small \
    --out /kaggle/working/runs/p1_pcen_on/snr_sweep.csv


In [ ]:
!cd /kaggle/working/srib/code && python -m src.eval_snr \
    --ckpt /kaggle/working/runs/p1_logmel/best.pt \
    --musan_dir data/musan_small \
    --out /kaggle/working/runs/p1_logmel/snr_sweep.csv


In [ ]:
import csv, json

print("=" * 62)
print("P0 CLEAN BASELINE")
print("=" * 62)
m = json.load(open("/kaggle/working/runs/p0/metrics.json"))
print(f"  best_val_acc = {m['best_val_acc']:.4f}  (trunk {m['trunk_params']/1e3:.1f}K)")

print("\n" + "=" * 62)
print("P1 PCEN ABLATION — clean val_acc (during training)")
print("=" * 62)
for name in ["p1_pcen_on", "p1_logmel"]:
    m = json.load(open(f"/kaggle/working/runs/{name}/metrics.json"))
    print(f"  {name:12s}  best = {m['best_val_acc']:.4f}")

print("\n" + "=" * 62)
print("P1 SNR SWEEP (MUSAN noise mixed at test time)")
print("=" * 62)
on = {float(r['snr_db']): float(r['val_acc'])
      for r in csv.DictReader(open("/kaggle/working/runs/p1_pcen_on/snr_sweep.csv"))}
off = {float(r['snr_db']): float(r['val_acc'])
       for r in csv.DictReader(open("/kaggle/working/runs/p1_logmel/snr_sweep.csv"))}
print(f"  {'SNR':>8s} | {'PCEN-ON':>8s} | {'log-Mel':>8s} | {'Δ (pp)':>8s}")
print(f"  {'-'*8} | {'-'*8} | {'-'*8} | {'-'*8}")
for s in sorted(on.keys(), reverse=True):
    label = "clean" if s >= 900 else f"{int(s)}dB"
    d = (on[s] - off[s]) * 100
    print(f"  {label:>8s} | {on[s]:>8.4f} | {off[s]:>8.4f} | {d:>+8.2f}")


In [ ]:
!tar czf /kaggle/working/all_results.tar.gz -C /kaggle/working runs
!ls -lh /kaggle/working/*.tar.gz
print("\nAll outputs in /kaggle/working/ — persist via Save & Run All.")
